# Knee 0.939 → 0.956: what moved the score, and what didn't

Fourteen scored submissions between 7 September and 15 October 2026.
The best public score reached **0.956** (rank **540 / 6,016** at the last snapshot). This notebook is the ledger behind
that number: which additions moved the displayed score, which ones made it worse, what was measured offline, and a few
leaderboard and engineering mechanics that cost me time.

**What I did.** Combined public checkpoints and inference pipelines into submissions, hardened them against the failures
that kill hidden-test reruns (GPU mismatch, NaN propagation, the 9-hour limit), found and fixed a component that had been
silently disabled in a popular public notebook, tested additions one at a time, and measured label quality offline on the
58 radiologist-labelled studies. The public notebooks and weights each submission builds on are listed in the table and
credited at the end.

All numbers below come from the tables in the code cells, and the asserts check the claims made in the text.
The notebook runs on CPU in a few seconds and needs no input data.

## TL;DR

- **Every visible step up came from a newly published trained model**, not from re-weighting: new CoAtNet readers (0.943),
  an independently trained 2.5D ConvNeXt reader (0.944), nartaa's CoAtNet-384 (0.950), heliosli's R7 model (0.956).
- **Three of my own additions on top of strong public bases went down**: a meniscus specialist (0.943 → 0.942),
  a public one-fold ConvNeXt (0.944 → 0.941), and an extra reader on top of the 0.956 base (same display, lower unrounded score).
- **On a plateau, rank is decided below the third decimal.** Submissions that all display 0.950 or 0.956 moved my rank
  by 100+ places. Kaggle's *auto-selection* markers are a free way to see which of your own equal-looking submissions is higher.
- **Medal cutoffs moved fast**: the bronze line went from 0.943 to 0.956 in 13 days, each time a new public model appeared.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# One row per scored submission (Kaggle submissions API, dates in UTC). "change" is what differed from the
# previous attempt; "source" credits the public notebook or model that the submission was built on.
subs = pd.DataFrame(json.loads(r"""[
 {"n": 1, "ref": 56067460, "utc": "2026-09-07 02:50", "score": 0.939, "change": "Fork of the public 'Bend the Knee to the Dinosaurs' stack", "source": "mattiaangeli"},
 {"n": 2, "ref": 56156616, "utc": "2026-09-11 02:15", "score": 0.939, "change": "Robustness fixes only (GPU probe, NaN guards, time budgets)", "source": "same stack"},
 {"n": 3, "ref": 56299768, "utc": "2026-09-17 08:26", "score": 0.941, "change": "Per-finding outer weights ('probe #22') instead of a flat 0.60", "source": "maverickss26"},
 {"n": 4, "ref": 56741006, "utc": "2026-10-01 01:58", "score": 0.943, "change": "New public stack with two new CoAtNet readers (D4, Global96)", "source": "romantamrazov / mattiaangeli"},
 {"n": 5, "ref": 56768021, "utc": "2026-10-02 07:00", "score": 0.942, "change": "Same stack, meniscus specialist actually running (see below)", "source": "renta0426 specialist"},
 {"n": 6, "ref": 56839986, "utc": "2026-10-05 02:22", "score": 0.944, "change": "Community stack + independently trained 2.5D ConvNeXt reader at 30%", "source": "goodpjw2008"},
 {"n": 7, "ref": 56872469, "utc": "2026-10-06 06:22", "score": 0.941, "change": "#6 + public one-fold ConvNeXt-Small M448 at 20%", "source": "mattiaangeli M448 fold 0"},
 {"n": 8, "ref": 56895584, "utc": "2026-10-07 00:42", "score": 0.945, "change": "#6 + meniscus specialist + slot fallback in the reader", "source": "sujanmajhisuzan"},
 {"n": 9, "ref": 56982524, "utc": "2026-10-09 01:01", "score": 0.950, "change": "nartaa CoAtNet-384 + goodpjw reader (two models only)", "source": "matterhorn3838 / nartaa"},
 {"n": 10, "ref": 56986573, "utc": "2026-10-09 02:40", "score": 0.950, "change": "Same two models, different per-finding weights", "source": "romantamrazov"},
 {"n": 11, "ref": 57124367, "utc": "2026-10-13 00:27", "score": 0.956, "change": "heliosli R7 ConvNeXt + nartaa CoAtNet-384, 50/50", "source": "heliosli / nartaa"},
 {"n": 12, "ref": 57163120, "utc": "2026-10-14 01:51", "score": 0.956, "change": "#11 + nartaa's second (224 px) checkpoint inside the Raptor half", "source": "varshithakolupuri"},
 {"n": 13, "ref": 57163177, "utc": "2026-10-14 01:53", "score": 0.956, "change": "#11 + goodpjw reader, per-finding weights", "source": "goodpjw2008 Part B"},
 {"n": 14, "ref": 57200677, "utc": "2026-10-15 01:00", "score": 0.956, "change": "heliosli's R7 + Raptor 384/224 recipe", "source": "heliosli"}
]"""))
subs["utc"] = pd.to_datetime(subs["utc"])

assert len(subs) == 14 and subs["ref"].is_unique
assert subs["score"].iloc[0] == 0.939 and subs["score"].max() == 0.956
display(subs.style.format({"score": "{:.3f}", "utc": lambda d: d.strftime("%m-%d %H:%M")}).hide(axis="index"))

## 1. The score trail

Each colour is a different public base. The pattern is the main lesson of this notebook: the line only steps up when the
base changes, i.e. when someone published a newly trained model.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(subs["n"], subs["score"], color="#9aa3ad", lw=1.2, zorder=1)
up = subs["score"] > subs["score"].cummax().shift(1).fillna(0)
ax.scatter(subs["n"], subs["score"], s=60, color=np.where(up, "#2B7A4B", "#B24C3E"), zorder=2)
for _, r in subs[up].iterrows():
    ax.annotate(f"{r.score:.3f}", (r.n, r.score), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax.set(title="Public score by submission (green = higher displayed best; #10, #12, #14 tied on display but ranked higher)",
       xlabel="submission #", ylabel="public macro AUC (displayed, 3 decimals)")
ax.set_xticks(subs["n"]); ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.show()

steps = subs[up][["n", "score", "change", "source"]].copy()
steps["gain"] = steps["score"].diff().fillna(0)
display(steps.style.format({"score": "{:.3f}", "gain": "{:+.3f}"}).hide(axis="index"))

## 2. What moved the displayed score

| step | what changed | gain | why it is "new information" |
|---|---|---|---|
| 0.939 → 0.941 | per-finding outer weights instead of a flat 0.60 | +0.002 | weights published after leaderboard probes by their author |
| 0.941 → 0.943 | two new CoAtNet readers (D4 depth zones, Global96) | +0.002 | newly trained models |
| 0.943 → 0.944 | an independently trained 2.5D ConvNeXt reader at 30% | +0.001 | trained from scratch, its own preprocessing |
| 0.945 → 0.950 | nartaa's CoAtNet-384 SWA (0.949 on its own) | +0.005 | one strong new model beat a 20+ model stack |
| 0.950 → 0.956 | heliosli's R7 ConvNeXt model | +0.006 | newly trained model |

The two largest jumps came from *fewer* models, not more: a single new CoAtNet (0.949 alone) replaced a stack of DINOv2,
RadImageNet, Raptor and four CoAtNet readers that together scored 0.943.

## 3. What did not work

Each of these was a single, pre-specified change on top of a base whose score was already known.

In [ ]:
negatives = pd.DataFrame([
    ["Meniscus specialist (10%) on the DINOsaur V5 stack", 0.943, 0.942, "displayed"],
    ["Public one-fold ConvNeXt-Small M448 at 20% on the 0.944 stack", 0.944, 0.941, "displayed"],
    ["goodpjw reader (per-finding weights) on the 0.956 R7 + Raptor base", 0.956, 0.956, "same display, lower unrounded (see section 4)"],
], columns=["addition", "base", "result", "how we know"])
negatives["delta"] = negatives["result"] - negatives["base"]
assert (negatives["delta"] <= 0).all()
display(negatives.style.format({"base": "{:.3f}", "result": "{:.3f}", "delta": "{:+.3f}"}).hide(axis="index"))

**The meniscus specialist had never actually run.** In the public DINOsaur V5 notebook the specialist captures the 336 px
DINO cache by wrapping `numpy.zeros` and keeping only allocations whose *immediate* caller is `build_cache`.
The fast base allocates that cache through a helper, `rsna_array()`: below 1 GiB it calls `np.zeros` *inside the helper*
(so the caller check sees the helper and drops it), and above 1 GiB, which is every hidden-test run, it allocates a disk
memmap and never calls `np.zeros` at all. The log line was `expected exactly one captured 6x12x336 cache/mask pair, got 0`,
followed by a silent fallback to the anchor. That is why "anchor" and "main" scored the same.
After wrapping `rsna_array` as well, the specialist ran end to end, and the score went **down** from 0.943 to 0.942.
A component that silently falls back can look neutral for weeks.

**The one-fold M448** is a public fallback for a private five-fold model. At a fixed 20% on top of 0.944 it scored 0.941.
If a time guard had stopped it, the result would have equalled 0.944 exactly, so the drop is real.

**An extra reader on top of the 0.956 base** displayed 0.956 like the base, but was not among the two auto-selection
candidates (next section), so its unrounded score was lower.

## 4. Leaderboard mechanics that mattered more than I expected

1. **Rank is decided by the unrounded score.** The leaderboard shows three decimals; on a plateau, hundreds of teams show
   the same number. Submissions #9 and #10 both display 0.950, but #10 moved me from rank 428 to 324.
2. **Exact ties go to the earlier submission.** Forks of the same public notebook produce byte-identical files, so on a
   popular release the order is set by submission time. Submitting a strong new public notebook early matters.
3. **Auto-selection markers reveal your own ordering.** If you have not selected final submissions, Kaggle marks the two it
   would pick. In my runs these were always the two highest unrounded scores, which told me that #13 was below #11 and #12
   without any extra submission.
4. **Public is 30% of the test set.** Differences below 0.001 between forks are unlikely to survive the private 70%.

In [ ]:
# Leaderboard snapshots downloaded during the competition (UTC). Medal lines use Kaggle's usual progression
# for 1,000+ teams (gold: top 10 + 0.2%, silver: top 5%, bronze: top 10%), so they are approximate.
snap = pd.DataFrame(json.loads(r"""[
 {"utc": "2026-10-02 02:11", "teams": 4810, "gold": 0.957, "silver": 0.944, "bronze": 0.943, "mine": 0.943, "rank": 887},
 {"utc": "2026-10-05 00:22", "teams": 5167, "gold": 0.958, "silver": 0.945, "bronze": 0.943, "mine": 0.943, "rank": 1136},
 {"utc": "2026-10-06 04:48", "teams": 5278, "gold": 0.958, "silver": 0.945, "bronze": 0.943, "mine": 0.944, "rank": 408},
 {"utc": "2026-10-09 00:53", "teams": 5542, "gold": 0.960, "silver": 0.950, "bronze": 0.950, "mine": 0.945, "rank": 887},
 {"utc": "2026-10-09 02:36", "teams": 5545, "gold": 0.960, "silver": 0.950, "bronze": 0.950, "mine": 0.950, "rank": 428},
 {"utc": "2026-10-09 05:08", "teams": 5552, "gold": 0.960, "silver": 0.950, "bronze": 0.950, "mine": 0.950, "rank": 324},
 {"utc": "2026-10-13 00:23", "teams": 5870, "gold": 0.962, "silver": 0.954, "bronze": 0.954, "mine": 0.950, "rank": 1313},
 {"utc": "2026-10-13 05:09", "teams": 5886, "gold": 0.962, "silver": 0.956, "bronze": 0.954, "mine": 0.956, "rank": 270},
 {"utc": "2026-10-14 01:47", "teams": 5953, "gold": 0.962, "silver": 0.956, "bronze": 0.956, "mine": 0.956, "rank": 728},
 {"utc": "2026-10-15 00:27", "teams": 6013, "gold": 0.963, "silver": 0.956, "bronze": 0.956, "mine": 0.956, "rank": 591},
 {"utc": "2026-10-15 06:25", "teams": 6016, "gold": 0.963, "silver": 0.956, "bronze": 0.956, "mine": 0.956, "rank": 540}
]"""))
snap["utc"] = pd.to_datetime(snap["utc"])
snap["top_pct"] = 100 * snap["rank"] / snap["teams"]

# claims made in the text
same = snap[(snap.mine == 0.950) & (snap.utc.dt.strftime("%m-%d") == "10-09")]   # submissions #9 and #10
assert same["rank"].iloc[0] - same["rank"].iloc[-1] == 104          # 428 -> 324 at a constant 0.950
assert snap["bronze"].iloc[0] == 0.943 and snap["bronze"].iloc[-1] == 0.956
assert (snap["utc"].iloc[-1] - snap["utc"].iloc[0]).days == 13

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
a1.step(snap["utc"], snap["bronze"], where="post", color="#C68B2C", label="bronze line")
a1.step(snap["utc"], snap["silver"], where="post", color="#7c8a99", label="silver line")
a1.step(snap["utc"], snap["mine"], where="post", color="#315A8A", lw=2, label="my best")
a1.set(title="Medal lines rose with every new public model", ylabel="public AUC (displayed)")
a1.legend(frameon=False); a1.grid(alpha=0.25)
a2.plot(snap["utc"], snap["rank"], marker="o", color="#315A8A")
a2.invert_yaxis(); a2.grid(alpha=0.25)
a2.set(title="My rank: the same displayed score can sit 100s of places apart", ylabel="rank (lower is better)")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()
display(snap.style.format({"gold": "{:.3f}", "silver": "{:.3f}", "bronze": "{:.3f}", "mine": "{:.3f}",
                           "top_pct": "{:.1f}%", "utc": lambda d: d.strftime("%m-%d %H:%M")}).hide(axis="index"))

## 5. Measured offline on the 58 labelled studies

Only 58 training studies carry radiologist labels. Two small findings came out of them; both are noisy (n = 58) and are
reported as observations, not as tuned recipes.

**a) The public multilingual report lexicon ranked findings better than the LLM label table** that overrides it in the
training code (`read_labels()` replaces lexicon labels with the table wherever the table has a row, i.e. 4,406 of 4,407
studies). Paired bootstrap over the same 58 studies:

In [ ]:
paired = pd.DataFrame(json.loads(r"""[
 {"finding": "Contusion", "lexicon": 0.856, "llm": 0.689, "lo": 0.054, "hi": 0.282},
 {"finding": "Effusion", "lexicon": 0.777, "llm": 0.652, "lo": 0.004, "hi": 0.241},
 {"finding": "ACL", "lexicon": 0.953, "llm": 0.868, "lo": 0.020, "hi": 0.159},
 {"finding": "MCL", "lexicon": 0.964, "llm": 0.906, "lo": 0.007, "hi": 0.111},
 {"finding": "Lateral Meniscus", "lexicon": 0.846, "llm": 0.792, "lo": -0.005, "hi": 0.117},
 {"finding": "Lateral OA", "lexicon": 0.839, "llm": 0.790, "lo": -0.029, "hi": 0.130},
 {"finding": "Medial Meniscus", "lexicon": 0.894, "llm": 0.867, "lo": -0.024, "hi": 0.117},
 {"finding": "Synovitis", "lexicon": 0.709, "llm": 0.691, "lo": -0.054, "hi": 0.138},
 {"finding": "Medial OA", "lexicon": 0.895, "llm": 0.860, "lo": -0.076, "hi": 0.150},
 {"finding": "Baker's", "lexicon": 0.940, "llm": 0.914, "lo": -0.015, "hi": 0.072},
 {"finding": "PF OA", "lexicon": 0.833, "llm": 0.821, "lo": -0.062, "hi": 0.102},
 {"finding": "Fracture", "lexicon": 0.833, "llm": 0.843, "lo": -0.077, "hi": 0.040}
]"""))
paired["delta"] = paired["lexicon"] - paired["llm"]
paired["verdict"] = np.where(paired["lo"] > 0, "lexicon better", np.where(paired["hi"] < 0, "LLM better", "not separable"))
assert (paired["delta"] > 0).sum() == 11 and (paired["verdict"] == "lexicon better").sum() == 4
assert (paired["verdict"] == "LLM better").sum() == 0
display(paired.style.format({c: "{:.3f}" for c in ["lexicon", "llm", "delta"]} | {"lo": "{:+.3f}", "hi": "{:+.3f}"})
        .hide(axis="index"))
print(f"lexicon ahead on {(paired.delta > 0).sum()}/12 findings, significantly on {(paired.verdict == 'lexicon better').sum()}; "
      f"macro {paired.lexicon.mean():.3f} vs {paired.llm.mean():.3f}")

**b) Most Synovitis training targets are not read from the report.** When a report never mentions synovitis, the lexicon
imputes it from the Effusion score. That back-off fires on **3,841 of 4,407 reports (87%)**, which fits Synovitis being
one of the weakest findings for every public model.

**c) A negative result: cross-finding post-processing.** Re-ranking a finding with a little of a related finding's score
(e.g. Synovitis with Effusion) did not help. Searching partner and weight on half of the 58 studies and scoring on the
other half, over 500 splits, *lost* 0.020 macro AUC on average: on 58 studies the search mostly finds noise.
The heads already share one backbone, so a neighbour's score adds little that the model has not seen.

In [ ]:
assert round(3841 / 4407, 3) == 0.872

## 6. Engineering notes (things that cost a run each)

- **A P100 cannot run the current image's PyTorch** (`sm_60` is not compiled in; `sm_70`+ only). `torch.cuda.is_available()`
  still returns `True`. Probe with a real kernel launch before trusting the GPU. If you push with the Kaggle CLI, set
  `"machine_shape": "NvidiaTeslaT4"` in `kernel-metadata.json`; without it my run was placed on a P100.
- **bf16 on a T4** (`sm_75`): one public `amp_for()` computed the compute capability and then returned bf16 anyway.
  Choose fp16 below `sm_80`.
- **Submission file selection**: when a notebook writes several CSVs, the submit dialog may pick an intermediate file
  (names starting with `_` sort first). Pick `submission.csv` explicitly, or submit with
  `kaggle competitions submit -k <kernel> -v <version> -f submission.csv`.
- **The 9-hour limit** includes everything appended after the base. Give every added stage its own deadline and a
  fallback that restores the base file, so a slow addition costs the addition, not the submission.
- **Kaggle CLI on a Japanese Windows locale**: older CLI versions read notebooks with the locale codec (`cp932`) and
  failed on UTF-8 characters. CLI 2.x reads them as UTF-8.

## 7. Limitations

- All outcomes are public-leaderboard scores (30% of the test set). The private ranking is unknown at the time of writing.
- Scores are displayed with three decimals; differences below 0.001 are inferred from rank changes and auto-selection only.
- The submissions form an observational sequence, not a controlled experiment; several rows changed more than one thing.
- The offline analyses use 58 studies and one model family; they indicate directions, not tuned settings.

## Credits

Everything above 0.939 stands on public work. Thank you to
**nartaa** (CoAtNet-384 / 224 SWA weights), **heliosli** (R7 ConvNeXt, OrthoFoundation, blend notebooks),
**goodpjw2008** (2.5D ConvNeXt MIL reader), **mattiaangeli** (Bend the Knee / Speedy Raptors, CoAtNet readers, M448),
**dreaddevelopment / Roman Tamrazov** (Raptor recipe and weights, DINOsaur notebooks), **matterhorn3838** (Velciraptor),
**varshithakolupuri** (the unused 224 px checkpoint), **sujanmajhisuzan** (Tri-Specialist Superstack),
**maverickss26** (restructured stack and probe #22), **jiweiliu** (fast 2×T4 inference), **renta0426** (meniscus specialist),
**pilkwang**, **prvsiyan**, **sofiaanjenje**, **antoinegg1**, **marwanmath**, **evgendvorkin** and everyone else who shared code
and weights in this competition.